# Dating RL v1: Tabular Q-Learning From Scratch

This notebook builds a small reinforcement-learning experiment using only standard Python plus NumPy. There is no Gymnasium, no PPO, no deep RL, no virtual environment, no database, and no API.

The goal is to compare three policies in a toy dating/conversation simulator:

- random actions
- hand-written rule-based actions
- learned tabular Q-learning actions

The main metric is `date_rate`: how often an episode ends in `date_success`.


## 1. Imports and Random Seeds

We only use `random` from standard Python and `numpy` for arrays, random sampling, and Q-table math.

In [23]:
import random

import numpy as np

np.random.seed(7)
random.seed(7)

## 2. State Space and Action Space

The state has 3 parts:

- `interest`: low, medium, high
- `stage`: opener, chat, date
- `tone`: cold, neutral, warm

That gives `3 x 3 x 3 = 27` discrete states.

The agent has 8 possible actions.

In [24]:
INTERESTS = ["low", "medium", "high"]
STAGES = ["opener", "chat", "date"]
TONES = ["cold", "neutral", "warm"]

ACTIONS = [
    "ask_question",
    "give_compliment",
    "share_story",
    "be_playful",
    "be_direct",
    "suggest_date",
    "slow_down",
    "end_chat",
]

N_INTEREST = len(INTERESTS)
N_STAGE = len(STAGES)
N_TONE = len(TONES)
N_STATES = N_INTEREST * N_STAGE * N_TONE
N_ACTIONS = len(ACTIONS)

print(f"Number of states: {N_STATES}")
print(f"Number of actions: {N_ACTIONS}")

Number of states: 27
Number of actions: 8


## 3. State Encoding Helpers

A Q-table needs integer state IDs. These helpers convert between a readable state tuple and a number from `0` to `26`.

In [25]:
def state_to_id(interest, stage, tone):
    """Convert a state tuple into one integer from 0 to 26."""
    return interest * (N_STAGE * N_TONE) + stage * N_TONE + tone


def id_to_state(state_id):
    """Convert one integer from 0 to 26 back into a state tuple."""
    interest = state_id // (N_STAGE * N_TONE)
    rest = state_id % (N_STAGE * N_TONE)
    stage = rest // N_TONE
    tone = rest % N_TONE
    return interest, stage, tone


def describe_state(state_id):
    interest, stage, tone = id_to_state(state_id)
    return f"interest={INTERESTS[interest]:6s} stage={STAGES[stage]:6s} tone={TONES[tone]}"


def clamp(value, low, high):
    return max(low, min(high, value))


for state_id in range(5):
    print(state_id, "->", describe_state(state_id))

0 -> interest=low    stage=opener tone=cold
1 -> interest=low    stage=opener tone=neutral
2 -> interest=low    stage=opener tone=warm
3 -> interest=low    stage=chat   tone=cold
4 -> interest=low    stage=chat   tone=neutral


## 4. Episode Reset

Each episode starts at the opener stage. Interest starts as low or medium, and tone starts cold or neutral.

In [26]:
def reset_episode():
    """Start with uncertain interest, early stage, and neutral/cold tone."""
    interest = np.random.choice([0, 1], p=[0.55, 0.45])
    stage = 0
    tone = np.random.choice([0, 1], p=[0.35, 0.65])
    return state_to_id(interest, stage, tone)


start_state = reset_episode()
print(start_state, describe_state(start_state))

1 interest=low    stage=opener tone=neutral


## 5. Hand-Coded Stochastic Simulator

This is the toy environment. It takes the current state and one action, then returns:

- the next state
- the reward
- whether the episode is done

The rules are intentionally simple and editable. Good timing tends to increase interest and tone. Pushy timing can lower both.

In [27]:
# =============================================================================
# REFACTORED simulator_step + new compute_reward
# Replace your old simulator_step cell with this entire block.
# =============================================================================
 
def compute_reward(old_state, new_state, outcome):
    """
    Reward depends ONLY on what happened to the state and the terminal outcome.
    It does NOT depend on which action was taken.
 
    This is the key fix: rewards are about OUTCOMES, not BEHAVIORS.
 
    old_state, new_state: (interest, stage, tone) tuples (ints)
    outcome: None, "date_success", "date_failed", "ghosted", "ended_by_agent"
    """
    # --- Terminal outcomes dominate everything else ---
    # These are big numbers compared to per-step shaping, so the agent
    # can never farm enough small rewards to beat a successful date.
    if outcome == "date_success":
        return 10.0
    if outcome == "ghosted":
        return -5.0
    if outcome == "date_failed":
        return -3.0
    if outcome == "ended_by_agent":
        return -1.0  # giving up is mildly bad
 
    # --- Per-step shaping: reward STATE PROGRESS ---
    # Interest going up = good. Tone getting warmer = good. Stage advancing = good.
    # Note: these are signed differences, so going BACKWARD gives NEGATIVE reward.
    # That naturally punishes actions that hurt rapport.
    old_i, old_s, old_t = old_state
    new_i, new_s, new_t = new_state
 
    interest_change = new_i - old_i   # -2 to +2
    stage_change = new_s - old_s      # -2 to +2
    tone_change = new_t - old_t       # -2 to +2
 
    # Weights: stage matters most (it's hardest to advance and crucial for date),
    # interest second, tone third (it's noisier and easier to swing).
    progress_reward = (
        1.5 * stage_change
        + 1.0 * interest_change
        + 0.5 * tone_change
    )
 
    # Per-step time cost. Without this, the agent can loop forever collecting
    # tiny shaping rewards. With this, every step costs something, so the
    # agent is forced to make real progress or end the episode.
    time_cost = -0.10
 
    return progress_reward + time_cost
 
 
def simulator_step(state_id, action):
    """
    Hand-coded stochastic simulator returning next_state, reward, done, outcome.
 
    KEY CHANGE: this function now only does DYNAMICS (how state changes).
    Rewards are computed at the END from the state change, by compute_reward().
    """
    interest, stage, tone = id_to_state(state_id)
    old_state = (interest, stage, tone)   # remember where we started
    done = False
    outcome = None
 
    # Small random shove on interest each step (some realism)
    noise = np.random.choice([-1, 0, 1], p=[0.10, 0.80, 0.10])
 
    # =========================================================================
    # DYNAMICS ONLY: each action changes the state. NO rewards in this block.
    # =========================================================================
    if action == 0:  # ask_question
        interest += np.random.choice([0, 1], p=[0.55, 0.45])
        tone += np.random.choice([0, 1], p=[0.50, 0.50])
 
    elif action == 1:  # give_compliment
        if tone >= 1:
            interest += 1   # works when tone is at least neutral
        else:
            tone -= 1       # backfires when tone is cold
 
    elif action == 2:  # share_story
        if stage >= 1:
            interest += np.random.choice([0, 1], p=[0.45, 0.55])
            tone += 1
        # else: nothing happens (sharing too early is just awkward, not harmful)
 
    elif action == 3:  # be_playful
        if tone == 2:
            interest += 1   # playfulness lands when tone is warm
        else:
            tone += np.random.choice([-1, 1], p=[0.45, 0.55])
 
    elif action == 4:  # be_direct
        if interest >= 1 and tone >= 1:
            stage += 1      # advances the stage when conditions are good
        else:
            interest -= 1   # backfires when too early
            tone -= 1
 
    elif action == 5:  # suggest_date
        # This is the only action that can end the episode in a "good" way.
        if interest == 2 and tone == 2:
            done = True
            outcome = "date_success"
        elif interest >= 1 and stage == 2:
            if np.random.random() < 0.50:
                done = True
                outcome = "date_success"
            else:
                done = True
                outcome = "date_failed"
        else:
            # Suggesting too early hurts and ends the episode
            done = True
            outcome = "date_failed"
 
    elif action == 6:  # slow_down
        tone += 1
        # Slowing down also gives a small chance of recovering interest
        if interest == 0 and np.random.random() < 0.3:
            interest += 1
 
    elif action == 7:  # end_chat
        done = True
        outcome = "ended_by_agent"
 
    # =========================================================================
    # Apply noise and clamp to legal ranges
    # =========================================================================
    interest = clamp(interest + noise, 0, N_INTEREST - 1)
    stage = clamp(stage, 0, N_STAGE - 1)
    tone = clamp(tone, 0, N_TONE - 1)
    next_state_id = state_to_id(interest, stage, tone)
    new_state = (interest, stage, tone)
 
    # =========================================================================
    # COMPUTE REWARD from state change + outcome (action-agnostic)
    # =========================================================================
    reward = compute_reward(old_state, new_state, outcome)
 
    return next_state_id, reward, done, outcome

## 6. Try One Simulator Step

This small test lets us see how one action changes the state. The simulator now also returns an `outcome`, which stays `None` unless the episode ended.


In [28]:
state = reset_episode()
action = ACTIONS.index("ask_question")
next_state, reward, done, outcome = simulator_step(state, action)

print("Before:", describe_state(state))
print("Action:", ACTIONS[action])
print("After: ", describe_state(next_state))
print("Reward:", round(reward, 3), "Done:", done, "Outcome:", outcome)


Before: interest=low    stage=opener tone=neutral
Action: ask_question
After:  interest=medium stage=opener tone=warm
Reward: 1.4 Done: False Outcome: None


## 7. Epsilon-Greedy Action Selection

The agent mostly chooses the best known action, but sometimes explores random actions.

- high epsilon = more exploration
- low epsilon = more exploitation

In [29]:
def choose_action(Q, state_id, epsilon):
    """Epsilon-greedy action choice."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[state_id]))

## 8. Rule-Based Policy

This is a hand-coded common-sense baseline. It does not learn. It gives the Q-learner a stronger comparison than random actions.

The rules are intentionally reasonable:

- suggest a date when interest and tone are both high
- avoid playful/direct moves when tone is cold
- warm up low-interest states
- ask questions early
- build rapport in the chat stage


In [30]:
def rule_based_action(state_id):
    """A simple common-sense policy with no learning."""
    interest, stage, tone = id_to_state(state_id)

    if interest == 2 and tone == 2:
        return ACTIONS.index("suggest_date")
    if stage == 2 and interest >= 1 and tone >= 1:
        return ACTIONS.index("suggest_date")
    if tone == 0:
        return ACTIONS.index("ask_question")
    if interest == 0:
        return ACTIONS.index("slow_down")
    if stage == 0:
        return ACTIONS.index("ask_question")
    if stage == 1 and tone == 2:
        return ACTIONS.index("be_playful")
    if stage == 1:
        return ACTIONS.index("share_story")
    return ACTIONS.index("be_direct")


for state_id in range(0, N_STATES, 5):
    action = rule_based_action(state_id)
    print(f"{describe_state(state_id)} -> {ACTIONS[action]}")


interest=low    stage=opener tone=cold -> ask_question
interest=low    stage=chat   tone=warm -> slow_down
interest=medium stage=opener tone=neutral -> ask_question
interest=medium stage=date   tone=cold -> ask_question
interest=high   stage=opener tone=warm -> suggest_date
interest=high   stage=date   tone=neutral -> suggest_date


## 9. Run One Episode

Episodes are variable length. They can end because the agent suggests a date, ends the chat, the conversation fizzles out, or the maximum step limit is reached.

When `learn=True`, this function also updates the Q-table. When `policy_fn` is provided, it uses that hand-coded policy instead of random or Q-table actions.


In [31]:
def run_episode(
    Q=None,
    policy_fn=None,
    epsilon=0.0,
    max_steps=25,
    learn=False,
    alpha=0.1,
    gamma=0.95,
):
    """Run one variable-length episode. If learn=True, update Q in place."""
    state_id = reset_episode()
    total_reward = 0.0

    for step in range(max_steps):
        if policy_fn is not None:
            action = policy_fn(state_id)
        elif Q is None:
            action = np.random.randint(N_ACTIONS)
        else:
            action = choose_action(Q, state_id, epsilon)

        next_state_id, reward, done, outcome = simulator_step(state_id, action)
        total_reward += reward

        if learn:
            best_next = np.max(Q[next_state_id])
            target = reward + (0.0 if done else gamma * best_next)
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        if done:
            return total_reward, step + 1, outcome

    return total_reward, max_steps, "max_steps"


reward, length, outcome = run_episode(Q=None)
print(f"One random episode: reward={reward:.3f}, length={length}, outcome={outcome}")


One random episode: reward=-1.000, length=1, outcome=ended_by_agent


## 10. Random and Rule-Based Baselines

Before training, we measure two baselines:

- `random`: chooses random actions
- `rule-based`: uses our hand-written common-sense rules

Both return average reward, average episode length, and date success rate.


In [32]:
def random_baseline(episodes=1000):
    rewards = []
    lengths = []
    n_dates = 0
    for _ in range(episodes):
        reward, length, outcome = run_episode(Q=None)
        rewards.append(reward)
        lengths.append(length)
        if outcome == "date_success":
            n_dates += 1
    return np.mean(rewards), np.mean(lengths), n_dates / episodes


def rule_based_baseline(episodes=1000):
    rewards = []
    lengths = []
    n_dates = 0
    for _ in range(episodes):
        reward, length, outcome = run_episode(policy_fn=rule_based_action)
        rewards.append(reward)
        lengths.append(length)
        if outcome == "date_success":
            n_dates += 1
    return np.mean(rewards), np.mean(lengths), n_dates / episodes


base_reward, base_length, base_date_rate = random_baseline()
rule_reward, rule_length, rule_date_rate = rule_based_baseline()

print(f"Random:     avg_reward={base_reward:.3f}  length={base_length:.2f}  date_rate={base_date_rate:.1%}")
print(f"Rule-based: avg_reward={rule_reward:.3f}  length={rule_length:.2f}  date_rate={rule_date_rate:.1%}")


Random:     avg_reward=0.084  length=4.04  date_rate=9.9%
Rule-based: avg_reward=11.745  length=5.49  date_rate=99.9%


## 11. Train Tabular Q-Learning

The Q-table has one row per state and one column per action.

Q-learning update:

`Q[state, action] = Q[state, action] + alpha * (target - Q[state, action])`

where the target is the reward plus the discounted value of the best next action.


In [33]:
def train_q_learning(episodes=8000):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        epsilon = max(0.05, 1.0 - episode / (episodes * 0.75))
        reward, _, _ = run_episode(Q=Q, epsilon=epsilon, learn=True)
        rewards.append(reward)

    return Q, rewards


Q, training_rewards = train_q_learning()
print("Q-table shape:", Q.shape)
print(f"Last 500 episode avg reward: {np.mean(training_rewards[-500:]):.3f}")


Q-table shape: (27, 8)
Last 500 episode avg reward: 13.834


## 12. Evaluate the Learned Policy

Now we run episodes with `epsilon=0`, meaning the agent always chooses the best learned action.

We compare the learned policy against both baselines using the headline metric: `date_rate`.


In [34]:
def evaluate_policy(Q, episodes=1000):
    rewards = []
    lengths = []
    n_dates = 0
    for _ in range(episodes):
        reward, length, outcome = run_episode(Q=Q, epsilon=0.0)
        rewards.append(reward)
        lengths.append(length)
        if outcome == "date_success":
            n_dates += 1
    return np.mean(rewards), np.mean(lengths), n_dates / episodes


learned_reward, learned_length, learned_date_rate = evaluate_policy(Q)

print(f"Random:     avg_reward={base_reward:.3f}  length={base_length:.2f}  date_rate={base_date_rate:.1%}")
print(f"Rule-based: avg_reward={rule_reward:.3f}  length={rule_length:.2f}  date_rate={rule_date_rate:.1%}")
print(f"Q-learned:  avg_reward={learned_reward:.3f}  length={learned_length:.2f}  date_rate={learned_date_rate:.1%}")


Random:     avg_reward=0.084  length=4.04  date_rate=9.9%
Rule-based: avg_reward=11.745  length=5.49  date_rate=99.9%
Q-learned:  avg_reward=14.748  length=5.83  date_rate=100.0%


## 13. Print the Learned Policy

For each of the 27 states, this prints the action with the highest Q-value.


In [35]:
def print_learned_policy(Q):
    print("\nLearned policy:")
    print("-" * 78)
    for state_id in range(N_STATES):
        best_action = int(np.argmax(Q[state_id]))
        value = Q[state_id, best_action]
        print(f"{state_id:02d} | {describe_state(state_id)} -> {ACTIONS[best_action]:14s} Q={value:6.2f}")


print_learned_policy(Q)


Learned policy:
------------------------------------------------------------------------------
00 | interest=low    stage=opener tone=cold -> slow_down      Q= 12.30
01 | interest=low    stage=opener tone=neutral -> slow_down      Q= 12.34
02 | interest=low    stage=opener tone=warm -> give_compliment Q= 12.24
03 | interest=low    stage=chat   tone=cold -> ask_question   Q=  1.26
04 | interest=low    stage=chat   tone=neutral -> share_story    Q=  8.85
05 | interest=low    stage=chat   tone=warm -> be_playful     Q= 11.40
06 | interest=low    stage=date   tone=cold -> ask_question   Q=  0.00
07 | interest=low    stage=date   tone=neutral -> ask_question   Q=  1.12
08 | interest=low    stage=date   tone=warm -> give_compliment Q=  3.80
09 | interest=medium stage=opener tone=cold -> slow_down      Q= 11.58
10 | interest=medium stage=opener tone=neutral -> slow_down      Q= 11.80
11 | interest=medium stage=opener tone=warm -> be_direct      Q= 11.94
12 | interest=medium stage=chat   tone

## 14. What to Edit Next

Good beginner experiments:

- change the rewards in `simulator_step`
- change the rule-based policy
- change the action list
- change `max_steps` in `run_episode`
- train for more or fewer episodes
- inspect the Q-table directly with `Q`


## 15. Kimi LLM State Classifier

This section converts a real dating-app message from the other person into the same `(interest, stage, tone)` state format used by the Q-table.

It uses Kimi through the OpenAI-compatible Python client. The notebook expects these environment variables to already be loaded earlier:

- `KIMI_API_KEY`
- `KIMI_BASE_URL`
- `KIMI_MODEL`

The classifier returns a tuple of three integers, for example `(2, 1, 2)` for high interest, chat stage, warm tone.


### 15.1 Label Mappings

The LLM returns string labels, but the Q-table uses integer state IDs. These dictionaries translate labels into the same integer indices used by `INTERESTS`, `STAGES`, and `TONES`.


In [ ]:
INTEREST_MAP = {"low": 0, "medium": 1, "high": 2}
STAGE_MAP = {"opener": 0, "chat": 1, "date": 2}
TONE_MAP = {"cold": 0, "neutral": 1, "warm": 2}


### 15.2 Classifier System Prompt

The prompt defines each label clearly so the model uses the same meaning as the simulator.


In [ ]:
CLASSIFIER_SYSTEM_PROMPT = """You are a careful classifier of dating-app conversation states. You will be given a single message from one person in a dating-app conversation. Your job is to label the conversation state along three dimensions, based ONLY on the most recent message and the optional history.

Output ONLY valid JSON with exactly these three fields:
- "interest": one of "low", "medium", "high"
- "stage": one of "opener", "chat", "date"
- "tone": one of "cold", "neutral", "warm"

Definitions:

INTEREST (how much the other person seems engaged with you):
- "low": short, dry replies; no questions back; no curiosity; one-word answers
- "medium": engaged but not effusive; some questions; reasonable response length
- "high": enthusiastic; multiple sentences; asks questions back; expresses positive emotion

STAGE (how far the conversation has progressed):
- "opener": first 1-2 exchanges, getting acquainted, basic introductions
- "chat": mid-conversation, established rapport, topics flowing, some back-and-forth
- "date": significant rapport built, flirtation present, OR discussing meeting up in person

TONE (the emotional register):
- "cold": terse, formal, low-affect, no warmth
- "neutral": cordial but not particularly warm; polite default
- "warm": friendly, jokes, banter, teasing, playful, emojis used genuinely

Output JSON only. No preamble, no explanation."""


### 15.3 Kimi Client Setup

This creates an OpenAI-compatible client pointed at Kimi. It raises a clear error if the required environment variables are missing.


In [ ]:
import json
import os

from openai import OpenAI

required_vars = ["KIMI_API_KEY", "KIMI_BASE_URL", "KIMI_MODEL"]
missing_vars = [name for name in required_vars if not os.getenv(name)]
if missing_vars:
    raise ValueError(f"Missing environment variable(s): {', '.join(missing_vars)}")

client = OpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url=os.getenv("KIMI_BASE_URL"),
)
KIMI_MODEL = os.getenv("KIMI_MODEL")


### 15.4 `classify(text, history=None)`

This function sends the latest message to Kimi, asks for JSON only, validates the returned labels, and converts the result into a tuple of three integers.


In [ ]:
def classify(text, history=None):
    """
    Classify one message into an (interest, stage, tone) integer state.

    text: most recent message from the OTHER person
    history: optional list of prior messages for context

    Returns: tuple of three ints
    Raises: ValueError if the LLM returns missing or invalid labels
    """
    if history:
        history_text = "\n".join(f"- {m}" for m in history)
        user_msg = f"Conversation history:\n{history_text}\n\nLatest message to classify:\n{text}"
    else:
        user_msg = f"Message to classify:\n{text}"

    response = client.chat.completions.create(
        model=KIMI_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content
    data = json.loads(raw)

    expected_keys = {"interest", "stage", "tone"}
    if set(data) != expected_keys:
        raise ValueError(
            f"Expected exactly {sorted(expected_keys)}, got {sorted(data)}. Raw output: {raw!r}"
        )

    if data["interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid interest label: {data['interest']!r}. Raw output: {raw!r}")
    if data["stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid stage label: {data['stage']!r}. Raw output: {raw!r}")
    if data["tone"] not in TONE_MAP:
        raise ValueError(f"Invalid tone label: {data['tone']!r}. Raw output: {raw!r}")

    return (
        INTEREST_MAP[data["interest"]],
        STAGE_MAP[data["stage"]],
        TONE_MAP[data["tone"]],
    )


### 15.5 Hand-Labeled Validation Examples

Run this cell after your Kimi environment variables are loaded. It compares the classifier against hand-labeled examples so you can quickly see whether its labels match your intuition.


In [ ]:
# Hand-labeled classifier validation examples.
test_cases = [
    ("hey", "low", "opener", "neutral"),
    ("lol that's actually really cute haha", "high", "chat", "warm"),
    ("k", "low", "opener", "cold"),
    ("yeah I love hiking too! where do you usually go?", "high", "chat", "warm"),
    ("wanna grab a drink this weekend?", "high", "date", "warm"),
    ("idk maybe", "low", "chat", "cold"),
    ("haha you're funny, what do you do for work?", "high", "chat", "warm"),
    ("i guess", "low", "chat", "cold"),
    ("OMG YES I LOVE THAT BAND", "high", "chat", "warm"),
    ("sure", "medium", "chat", "neutral"),
]

print(f"{'Message':<55} {'Expected':<25} {'Got':<25} {'Match'}")
print("-" * 115)

n_correct = 0
for msg, exp_i, exp_s, exp_t in test_cases:
    got_i, got_s, got_t = classify(msg)
    expected = f"({exp_i}, {exp_s}, {exp_t})"
    got = f"({INTERESTS[got_i]}, {STAGES[got_s]}, {TONES[got_t]})"
    match = (
        INTERESTS[got_i] == exp_i and
        STAGES[got_s] == exp_s and
        TONES[got_t] == exp_t
    )
    if match:
        n_correct += 1
    msg_display = msg if len(msg) <= 50 else msg[:47] + "..."
    print(f"{msg_display:<55} {expected:<25} {got:<25} {'yes' if match else 'no'}")

print(f"\nAccuracy: {n_correct}/{len(test_cases)} = {n_correct / len(test_cases):.0%}")
